pip install requests wikitextparser re os json

In [ ]:
%pip install requests wikitextparser
# If %pip fails (uv-managed venv): run in terminal → uv pip install requests wikitextparser

---
# Data Extraction from Terraria Wiki

In [1]:
import requests
import wikitextparser as wtp
import re
import os
import json
from kb_extraction import get_clean_terraria_wiki
from json_extract import save_to_manifest

## Guide:Getting Started

In [2]:
page_title = "Guide:Getting_started"
content = get_clean_terraria_wiki(page_title)
source_url = "https://terraria.wiki.gg/wiki/Guide:Getting_started"
game_state = "Pre-Hardmode"

In [3]:
with open("getting_started.txt", "w", encoding="utf-8") as file:
    file.write(content)

In [4]:
save_to_manifest(page_title, content, source_url, game_state)

Successfully saved manifest to: knowledge_base/processed/guide_getting_started.json


## Crafting 101

In [5]:
page_title = "Guide:Crafting_101"
content = get_clean_terraria_wiki(page_title)
source_url = "https://terraria.wiki.gg/wiki/Guide:Crafting_101"
game_state = "Pre-Hardmode"

In [6]:
with open("crafting101.txt", "w") as file:
    file.write(content)

In [7]:
save_to_manifest(page_title, content, source_url, game_state)

Successfully saved manifest to: knowledge_base/processed/guide_crafting_101.json


## Class Setups

In [8]:
page_title = "Guide:Class_setups"
content = get_clean_terraria_wiki(page_title)
source_url = "https://terraria.wiki.gg/wiki/Guide:Class_setups"
game_state = "Pre-Hardmode"

In [9]:
with open("class_setups.txt", "w") as file:
    file.write(content)

In [10]:
save_to_manifest(page_title, content, source_url, game_state)

Successfully saved manifest to: knowledge_base/processed/guide_class_setups.json


## Pre-Hardmode Walkthrough

In [11]:
page_title = "Guide:Walkthrough"
content = get_clean_terraria_wiki(page_title)
source_url = "https://terraria.wiki.gg/wiki/Guide:Walkthrough"
game_state = "Pre-Hardmode"

In [12]:
with open("Pre-Hardmode_Walkthrough.txt", "w") as file:
    file.write(content)

In [13]:
save_to_manifest(page_title, content, source_url, game_state)

Successfully saved manifest to: knowledge_base/processed/guide_walkthrough.json


## Getting started with hardmode

In [14]:
page_title = "Guide:Getting_started_with_Hardmode"
content = get_clean_terraria_wiki(page_title)
source_url = "https://terraria.wiki.gg/wiki/Guide:Getting_started_with_Hardmode"
game_state = "Hardmode"

In [15]:
with open("Getting_started_with_Hardmode.txt", "w") as file:
    file.write(content)

In [16]:
save_to_manifest(page_title, content, source_url, game_state)

Successfully saved manifest to: knowledge_base/processed/guide_getting_started_with_hardmode.json


## Hardmode walkthrough

In [17]:
page_title = "Guide:Walkthrough/Hardmode"
content = get_clean_terraria_wiki(page_title)
source_url = "https://terraria.wiki.gg/wiki/Guide:Walkthrough/Hardmode"
game_state = "Hardmode"

In [18]:
with open("Hardmode.txt", "w") as file:
    file.write(content)

In [19]:
save_to_manifest(page_title, content, source_url, game_state)

Successfully saved manifest to: knowledge_base/processed/guide_walkthrough_hardmode.json


## Practical Tips

In [20]:
page_title = "Guide:Practical_tips"
content = get_clean_terraria_wiki(page_title)
source_url = "https://terraria.wiki.gg/wiki/Guide:Practical_tips"
game_state = "Hardmode"

In [21]:
with open("Practical_tips.txt", "w") as file:
    file.write(content)

In [22]:
save_to_manifest(page_title, content, source_url, game_state)

Successfully saved manifest to: knowledge_base/processed/guide_practical_tips.json


## Bosses

In [24]:
page_title = "Bosses"
content = get_clean_terraria_wiki(page_title)
source_url = "https://terraria.wiki.gg/wiki/Bosses"
game_state = "Hardmode"

In [25]:
with open("Practical_tips.txt", "w") as file:
    file.write(content)

In [26]:
save_to_manifest(page_title, content, source_url, game_state)

Successfully saved manifest to: knowledge_base/processed/bosses.json


---
# Indexing

In [ ]:
from langchain_community.document_loaders import DirectoryLoader, JSONLoader

folder_path = "./knowledge_base/processed"
loader = DirectoryLoader(
    path=folder_path, 
    glob="*.json",
    loader_cls=JSONLoader,
    loader_kwargs = {
        "jq_schema": ".content",
        "text_content": False
    }
)

docs = loader.load()

print(f"Loaded {len(docs)} files.")
print(f"Snippet of first file: {docs[0].page_content[:50]}...")

In [ ]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=50
)

chunks = splitter.split_documents(docs)

print(f"Number of chunks created: {len(chunks)}")
print()

print("--- Chunk 0 (inspect before storing!) ---")
print(chunks[0].page_content)
print()
print("--- Chunk 1 ---")
print(chunks[1].page_content)

In [ ]:
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS

embeddings = HuggingFaceEmbeddings(
    model_name="all-MiniLM-L6-v2"
)

print("Embedding model loaded: all-MiniLM-L6-v2 (384-dimensional)")
print("Embedding all chunks into the vector database...")
db = FAISS.from_documents(chunks, embeddings)

print(f"Vector database created with {len(chunks)} vectors")

In [ ]:
db.save_local("faiss_index")
print("FAISS index saved to 'faiss_index/'")
print()
print("To reload later (skip the embedding step entirely):")
print("  db = FAISS.load_local('faiss_index', embeddings, allow_dangerous_deserialization=True)")

In [ ]:
retriever = db.as_retriever(
    search_kwargs={"k": 3}
)

print("Retriever created (k=3)")

In [ ]:
test_query = "How to craft items?"
retrieved_docs = retriever.invoke(test_query)

print(f"Query: {test_query}")
print(f"Number of chunks retrieved: {len(retrieved_docs)}")
print()
for i, doc in enumerate(retrieved_docs):
    print(f"--- Retrieved Chunk {i+1} ---")
    print(doc.page_content)
    print()

---
# Generation

pip install langchain-groq python-dotenv

In [ ]:
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS

embeddings = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")
db = FAISS.load_local("faiss_index", embeddings, allow_dangerous_deserialization=True)
retriever = db.as_retriever(search_kwargs={"k": 3})

print("Retriever loaded from faiss_index/")

In [ ]:
from langchain_groq import ChatGroq
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser
from dotenv import load_dotenv
import os

load_dotenv()
GROQ_API_KEY = os.getenv('GROQ_API_KEY')

llm = ChatGroq(
    model='llama-3.1-8b-instant',
    api_key=GROQ_API_KEY
)

def format_docs(docs):
    return '\n\n'.join(doc.page_content for doc in docs)

# Covers: KB lookup, class info, frustrated tone, off-topic,
#         tone-deaf failure (v2), Hardmode leakage (v2),
#         over-refusal (v2), multi-part question (v3)
test_queries = [
    'How do I craft a Work Bench?',
    'What class should I pick?',
    'ugh i keep dying to the Eye of Cthulhu',
    'Is Minecraft better than Terraria?',
    'ugh why do i keep dying to skeletons',
    'What ore should I mine?',
    'Is the Crimson dangerous?',
    'What class should I pick and how do I beat Skeletron?',
]

print('LLM loaded: llama-3.1-8b-instant (Groq)')

---
## Prompt v1 — Baseline
Bare prompt with no rules. LLM answers freely, ignoring KB boundaries.

**Known issues:** hallucinated items, no KB grounding, off-topic answered, inconsistent length.

In [ ]:
v1_prompt = ChatPromptTemplate.from_messages([
    ('system', (
        'You are a Terraria assistant. Use the context below to answer the question.\n\n'
        'Context: {retrieved_chunks}'
    )),
    ('human', '{question}')
])

v1_chain = (
    {'retrieved_chunks': retriever | format_docs, 'question': RunnablePassthrough()}
    | v1_prompt
    | llm
    | StrOutputParser()
)

print('=== Prompt v1 Results ===\n')
for query in test_queries:
    print(f'Q: {query}')
    print(f'A: {v1_chain.invoke(query)}')
    print()

---
## Prompt v2 — KB Constraint + Length Rule (Iteration 1)
Added KB-only rule, 120-word limit, structured format, and off-topic refusal.

**Fixed:** hallucinations, off-topic, inconsistent length.  
**Still broken:** tone-deaf on frustrated players, Hardmode content leakage, over-refusal on indirect questions.

In [ ]:
v2_system = (
    'You are TerrariaBot, an assistant for Terraria 1.4.4.\n\n'
    'Rules:\n'
    '1. Answer ONLY using information in the Context below.\n'
    '2. If the answer is not in the Context, say: "I don\'t have that info — check terraria.wiki.gg."\n'
    '3. Lead with a direct answer. Add detail after. Use bullet points for 3+ item lists.\n'
    '4. Keep responses under 120 words unless listing steps or items.\n'
    '5. Do not answer questions unrelated to Terraria.\n\n'
    'Context: {retrieved_chunks}'
)

v2_prompt = ChatPromptTemplate.from_messages([
    ('system', v2_system),
    ('human', '{question}')
])

v2_chain = (
    {'retrieved_chunks': retriever | format_docs, 'question': RunnablePassthrough()}
    | v2_prompt
    | llm
    | StrOutputParser()
)

print('=== Prompt v2 Results ===\n')
for query in test_queries:
    print(f'Q: {query}')
    print(f'A: {v2_chain.invoke(query)}')
    print()

---
## Prompt v3 — Few-Shot + Tone Matching + Edge Cases (Iteration 2)
Added few-shot examples, tone matching, and explicit edge case handling.

**Fixed:** frustrated tone, off-topic, indirect question over-refusal.  
**Still broken:** verbatim quoting of long chunks, multi-part questions get uneven answers.

In [ ]:
v3_system = (
    'You are TerrariaBot, a friendly assistant for Terraria 1.4.4.\n\n'
    'Rules:\n'
    '1. Answer ONLY using information in the Context below.\n'
    '2. If the answer is missing, say: "I don\'t have that detail — terraria.wiki.gg is a great next step."\n'
    '3. Lead with a direct answer. Use bullet points for 3+ item lists.\n'
    '4. Match the player\'s tone: be brief and encouraging for frustrated players; be thorough for detailed questions.\n'
    '5. For off-topic questions reply: "I only know Terraria — ask me about crafting, bosses, classes, or survival!"\n'
    '6. Never reference items or mechanics absent from the provided Context.\n\n'
    'Examples:\n\n'
    'User: How do I craft a Work Bench?\n'
    'Assistant: Craft a Work Bench with 10 Wood — no crafting station needed, just open your inventory. '
    'Place it on flat ground to unlock recipes for furniture, weapons, and walls.\n\n'
    'User: What weapons are good for a ranged build early on?\n'
    'Assistant: Good early ranged options include:\n'
    '- Wooden Bow + Flaming Arrows — reliable, also lights up caves\n'
    '- Minishark — fast fire rate, bought from the Arms Dealer\n'
    '- Boomstick — high damage, found in Jungle Chests\n'
    'Pair these with an Archery Potion for extra damage.\n\n'
    'User: ugh i keep dying to the Eye of Cthulhu this is so hard\n'
    'Assistant: Hang in there! Make sure you have at least 200 HP and a flat arena with platforms before the fight. '
    'Keep moving and dodge its dash in the second phase — a bow or shuriken works well for both phases.\n\n'
    'User: Is Fortnite better than Terraria?\n'
    'Assistant: I only know Terraria — ask me about crafting, bosses, classes, or survival!\n\n'
    'Context: {retrieved_chunks}'
)

v3_prompt = ChatPromptTemplate.from_messages([
    ('system', v3_system),
    ('human', '{question}')
])

v3_chain = (
    {'retrieved_chunks': retriever | format_docs, 'question': RunnablePassthrough()}
    | v3_prompt
    | llm
    | StrOutputParser()
)

print('=== Prompt v3 Results ===\n')
for query in test_queries:
    print(f'Q: {query}')
    print(f'A: {v3_chain.invoke(query)}')
    print()